# Feature Engineering and Preprocessing

This notebook performs:

1. Load Cleaned Dataset
2. Create New Features
3. Identify Important Features
4. Prepare Categorical Features
5. Prepare Numerical Features
6. Encode Categorical Features
7. Scale Numerical Features
8. Prepare Final Dataset for Model Training

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [2]:
DATA_PATH = "../outputs/cleaned_data.csv"

OUTPUT_PATH = "../outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Dataset Shape: {df.shape}")

df.head()

Dataset loaded successfully.
Dataset Shape: (12205, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
df["TotalPages"] = (
    df["Administrative"] +
    df["Informational"] +
    df["ProductRelated"]
)

print("TotalPages feature created.")

TotalPages feature created.


In [5]:
df["TotalDuration"] = (
    df["Administrative_Duration"] +
    df["Informational_Duration"] +
    df["ProductRelated_Duration"]
)

print("TotalDuration feature created.")

TotalDuration feature created.


In [6]:
df["AveragePageDuration"] = np.where(
    df["TotalPages"] > 0,
    df["TotalDuration"] / df["TotalPages"],
    0
)

print("AveragePageDuration feature created.")

AveragePageDuration feature created.


In [7]:
df["EngagementScore"] = (
    df["TotalPages"] *
    df["AveragePageDuration"]
)

print("EngagementScore feature created.")

EngagementScore feature created.


In [8]:
feature_engineering_summary = pd.DataFrame({
    "New Feature": [
        "TotalPages",
        "TotalDuration",
        "AveragePageDuration",
        "EngagementScore"
    ],
    
    "Purpose": [
        "Measures total pages visited",
        "Measures total browsing duration",
        "Measures average time spent per page",
        "Measures overall website engagement"
    ]
})

feature_engineering_summary

,New Feature,Purpose
0,TotalPages,Measures total pages visited
1,TotalDuration,Measures total browsing duration
2,AveragePageDuration,Measures average time spent per page
3,EngagementScore,Measures overall website engagement


In [9]:
df["Revenue"] = df["Revenue"].astype(int)

print(df["Revenue"].value_counts())

Revenue
0    10297
1     1908
Name: count, dtype: int64


In [10]:
X = df.drop("Revenue", axis=1)

y = df["Revenue"]

print(f"Feature Shape: {X.shape}")
print(f"Target Shape: {y.shape}")

Feature Shape: (12205, 21)
Target Shape: (12205,)


In [11]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'TotalPages', 'TotalDuration', 'AveragePageDuration', 'EngagementScore']

Categorical Features:
['Month', 'VisitorType', 'Weekend']


In [12]:
numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ],
    
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-Test Split Completed")

print(f"\nTraining Data: {X_train.shape}")
print(f"Testing Data: {X_test.shape}")

Train-Test Split Completed

Training Data: (9764, 21)
Testing Data: (2441, 21)


In [15]:
split_distribution = pd.DataFrame({
    "Dataset": [
        "Full Dataset",
        "Training Dataset",
        "Testing Dataset"
    ],
    
    "Purchase Percentage": [
        round(y.mean() * 100, 2),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

split_distribution

,Dataset,Purchase Percentage
0,Full Dataset,15.63
1,Training Dataset,15.63
2,Testing Dataset,15.65


In [16]:
train_data = X_train.copy()

train_data["Revenue"] = y_train.values

test_data = X_test.copy()

test_data["Revenue"] = y_test.values

train_data.to_csv(
    "../outputs/train_data.csv",
    index=False
)

test_data.to_csv(
    "../outputs/test_data.csv",
    index=False
)

print("Train and Test datasets saved successfully.")

Train and Test datasets saved successfully.


In [17]:
preprocessing_summary = pd.DataFrame({
    "Step": [
        "Numerical Features",
        "Categorical Features",
        "Scaling",
        "Encoding",
        "Train-Test Split"
    ],
    
    "Method Used": [
        f"{len(numerical_features)} features",
        f"{len(categorical_features)} features",
        "StandardScaler",
        "OneHotEncoder",
        "Stratified 80-20 Split"
    ],
    
    "Reason": [
        "Identify numerical variables",
        "Identify categorical variables",
        "Normalize numerical feature scales",
        "Convert categorical data to numerical format",
        "Maintain target class distribution"
    ]
})

preprocessing_summary

,Step,Method Used,Reason
0,Numerical Features,18 features,Identify numerical variables
1,Categorical Features,3 features,Identify categorical variables
2,Scaling,StandardScaler,Normalize numerical feature scales
3,Encoding,OneHotEncoder,Convert categorical data to numerical format
4,Train-Test Split,Stratified 80-20 Split,Maintain target class distribution


In [18]:
joblib.dump(
    preprocessor,
    "../outputs/preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [19]:
print("=" * 60)
print("NOTEBOOK 02 COMPLETED")
print("=" * 60)

print("\nFeature Engineering Completed")

print("\nNew Features Created:")
print("- TotalPages")
print("- TotalDuration")
print("- AveragePageDuration")
print("- EngagementScore")

print("\nPreprocessing Strategy:")
print("- Numerical Data → StandardScaler")
print("- Categorical Data → OneHotEncoder")
print("- Data Split → Stratified 80/20")

print("\nNext Step:")
print("Model Training and Validation")

NOTEBOOK 02 COMPLETED

Feature Engineering Completed

New Features Created:
- TotalPages
- TotalDuration
- AveragePageDuration
- EngagementScore

Preprocessing Strategy:
- Numerical Data → StandardScaler
- Categorical Data → OneHotEncoder
- Data Split → Stratified 80/20

Next Step:
Model Training and Validation
